#GLiNER: распознавание сущностей с произвольной схемой типов



## 1. Установка и настройки

В учебном примере используется небольшая англоязычная модель `gliner-community/gliner_small-v2.5`. Русскоязычные тексты рассматриваются в конце отдельно: для них предпочтительна многоязычная модель.

In [1]:
%pip install -q "gliner>=0.2.0"

In [2]:
from gliner import GLiNER
import pandas as pd
import matplotlib.pyplot as plt

# Настраиваемые параметры
MODEL_NAME = "gliner-community/gliner_small-v2.5"
THRESHOLD = 0.4

model = GLiNER.from_pretrained(MODEL_NAME)
print(f"Модель загружена: {MODEL_NAME}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Модель загружена: gliner-community/gliner_small-v2.5


## 2. Первое распознавание: стандартные типы

Модель получает **текст** и **список текстовых обозначений типов**. Список не обязательно совпадает с фиксированной схемой разметки из обучающего корпуса.

In [3]:

text = (
    "Amazon announced a $750 million investment in a new "
    "artificial intelligence research center in Cambridge, "
    "Massachusetts. The project will be led by Dr. Sarah Mitchell, "
    "who previously worked at Microsoft. "
    "Meanwhile, an environmental organization called Amazon Watch "
    "criticized the company's expansion near the Amazon River. "
    "The research center is expected to open in September 2027, "
    "although the final investment amount has not been confirmed."
)

labels = [
    "person",
    "organization",
    "location",
    "date",
    "money",
]

entities = model.predict_entities(text, labels, threshold=THRESHOLD,)

display(pd.DataFrame(entities, columns=["text", "label", "start", "end", "score"],))

,text,label,start,end,score
0,Amazon,organization,0,6,0.984191
1,$750 million,money,19,31,0.998007
2,Cambridge,location,95,104,0.916116
3,Massachusetts,location,106,119,0.919239
4,Dr. Sarah Mitchell,person,148,166,0.994646
5,Microsoft,organization,193,202,0.990812
6,Amazon Watch,organization,252,264,0.997164
7,Amazon River,location,309,321,0.984926
8,September 2027,date,366,380,0.999127


### Проверяем границы упоминаний

В выходных данных `start` и `end` задают начало и конец символьного интервала в исходной строке: `text[start:end]`. Это важно для последующего сохранения разметки и вычисления метрик.

In [4]:
for ent in entities:
    fragment = text[ent["start"]:ent["end"]]
    print(f"{fragment!r:30} | {ent['label']:15} | {ent['score']:.3f}")
    assert fragment == ent["text"], "Несовпадение границ и текста упоминания"
print("Проверка границ завершена")

'Amazon'                       | organization    | 0.984
'$750 million'                 | money           | 0.998
'Cambridge'                    | location        | 0.916
'Massachusetts'                | location        | 0.919
'Dr. Sarah Mitchell'           | person          | 0.995
'Microsoft'                    | organization    | 0.991
'Amazon Watch'                 | organization    | 0.997
'Amazon River'                 | location        | 0.985
'September 2027'               | date            | 0.999
Проверка границ завершена


## 3. Произвольные типы без дообучения

Изменим постановку задачи, **не меняя весов модели**. Например, вместо общего типа `money` ищем `investment amount`, а вместо `person` — `executive`. Новые названия типов могут изменить и множество найденных упоминаний.

In [5]:

custom_labels = [
    "technology company",
    "environmental organization",
    "research center",
    "researcher",
    "financial investment",
    "geographical feature",
]
custom_entities = model.predict_entities(text, custom_labels, threshold=0.3,)
pd.DataFrame(custom_entities, columns=["text", "label", "start", "end", "score"],)


,text,label,start,end,score
0,Amazon,technology company,0,6,0.997247
1,$750 million,financial investment,19,31,0.992335
2,artificial intelligence research center,research center,52,91,0.693410
3,Dr. Sarah Mitchell,researcher,148,166,0.991492
4,Microsoft,technology company,193,202,0.997127
5,Amazon,technology company,252,258,0.970528
6,Amazon River,geographical feature,309,321,0.972434


## 4. Чувствительность к формулировке типов

Сравним короткие и более подробные названия. Это **эксперимент**, а не гарантия, что подробные названия всегда лучше.

In [6]:
label_sets = {
    "Общие типы": [
        "organization",
        "person",
        "location",
        "money",
    ],
    "Специализированные типы": [
        "technology company",
        "environmental organization",
        "research center",
        "research scientist",
        "investment amount",
        "river",
    ],
    "Альтернативные формулировки": [
        "corporation",
        "nonprofit organization",
        "research facility",
        "scientist",
        "financial commitment",
        "natural geographical feature",
    ],
}

rows = []

for scheme_name, current_labels in label_sets.items():
    predictions = model.predict_entities(text, current_labels, threshold=0.3,)

    for entity in predictions:
        rows.append({
            "Схема": scheme_name,
            "Упоминание": entity["text"],
            "Тип": entity["label"],
            "Оценка": round(entity["score"], 3),
        })

comparison_df = pd.DataFrame(rows, columns=["Схема", "Упоминание", "Тип", "Оценка"])
comparison_df


,Схема,Упоминание,Тип,Оценка
0,Общие типы,Amazon,organization,0.982
1,Общие типы,$750 million,money,0.999
2,Общие типы,Cambridge,location,0.937
3,Общие типы,Massachusetts,location,0.926
4,Общие типы,Dr. Sarah Mitchell,person,0.995
5,Общие типы,Microsoft,organization,0.989
6,Общие типы,Amazon Watch,organization,0.997
7,Общие типы,Amazon River,location,0.982
8,Специализированные типы,Amazon,technology company,0.997
9,Специализированные типы,$750 million,investment amount,0.995


## 5. Порог принятия предсказаний

Порог регулирует, какие предсказания сохраняются. С уменьшением порога обычно возрастает число принятых гипотез, однако это само по себе не доказывает улучшение полноты или ухудшение точности — для этого нужна эталонная разметка. Оценка `score` не является автоматически калиброванной вероятностью корректности.

In [7]:

predictions = model.predict_entities(
    text,
    custom_labels,
    threshold=0.01,
)

confidence_df = pd.DataFrame(
    [
        {
            "Упоминание": e["text"],
            "Тип": e["label"],
            "Оценка": round(e["score"], 4),
            "При пороге 0.3": e["score"] >= 0.3,
            "При пороге 0.8": e["score"] >= 0.8,
        }
        for e in predictions
    ]
)

if not confidence_df.empty:
    confidence_df = confidence_df.sort_values("Оценка")

confidence_df


,Упоминание,Тип,Оценка,При пороге 0.3,При пороге 0.8
6,environmental organization,environmental organization,0.0710,False,False
9,research center,research center,0.0833,False,False
10,September 2027,financial investment,0.1556,False,False
3,Cambridge,geographical feature,0.1692,False,False
2,artificial intelligence research center,research center,0.6934,True,False
7,Amazon,technology company,0.9705,True,True
8,Amazon River,geographical feature,0.9724,True,True
4,Dr. Sarah Mitchell,researcher,0.9915,True,True
1,$750 million,financial investment,0.9923,True,True
5,Microsoft,technology company,0.9971,True,True


## 6. Предметная схема: международная торговля

Список меток можно ориентировать на предметную область. Важно различать **тип сущности** и **отношение/событие**: например, введение санкций — событие, а упоминание санкционной меры — возможная сущность. Границы и состав схемы определяет задача разметки.

In [8]:

trade_text = """
The European Commission proposed additional restrictions  on imports of Russian steel and aluminum products. However, the proposal does not cover medical equipment or agricultural fertilizers. Several European manufacturers warned that the measures could affect supply chains in Germany and Poland.
Meanwhile, India and the United Arab Emirates discussed possible amendments to their Comprehensive Economic Partnership Agreement (CEPA). Officials emphasized that the negotiations had not resulted in a new treaty. The two countries also considered reducing tariffs on pharmaceutical products and processed food.
In a separate development, the United States announced new export controls on advanced semiconductor equipment and certain artificial intelligence chips.
The restrictions were introduced under updated export control regulations.
"""


In [9]:

trade_labels = [
    "country",
    "international organization",
    "trade agreement",
    "trade restriction",
    "product category",
    "legal regulation",
]

trade_entities = model.predict_entities(trade_text, trade_labels, threshold=0.3,)

display(pd.DataFrame(trade_entities,columns=["text", "label", "start", "end", "score"],)
)
trade_entities_08 = model.predict_entities(trade_text, trade_labels, threshold=0.8)
print(trade_text)
display(pd.DataFrame(trade_entities_08, columns=["text", "label", "start", "end", "score"]))

,text,label,start,end,score
0,European Commission,international organization,5,24,0.985581
1,Russian,country,73,80,0.742213
2,aluminum,product category,91,99,0.619940
3,medical equipment,product category,147,164,0.935581
4,agricultural fertilizers,product category,168,192,0.899749
5,European manufacturers,international organization,202,224,0.435840
6,Germany,country,280,287,0.999472
7,Poland,country,292,298,0.999307
8,India,country,311,316,0.999597
9,United Arab Emirates,country,325,345,0.999228



The European Commission proposed additional restrictions  on imports of Russian steel and aluminum products. However, the proposal does not cover medical equipment or agricultural fertilizers. Several European manufacturers warned that the measures could affect supply chains in Germany and Poland.
Meanwhile, India and the United Arab Emirates discussed possible amendments to their Comprehensive Economic Partnership Agreement (CEPA). Officials emphasized that the negotiations had not resulted in a new treaty. The two countries also considered reducing tariffs on pharmaceutical products and processed food.
In a separate development, the United States announced new export controls on advanced semiconductor equipment and certain artificial intelligence chips.
The restrictions were introduced under updated export control regulations.



,text,label,start,end,score
0,European Commission,international organization,5,24,0.985581
1,medical equipment,product category,147,164,0.935581
2,agricultural fertilizers,product category,168,192,0.899749
3,Germany,country,280,287,0.999472
4,Poland,country,292,298,0.999307
5,India,country,311,316,0.999597
6,United Arab Emirates,country,325,345,0.999228
7,CEPA,trade agreement,431,435,0.908482
8,pharmaceutical products,product category,569,592,0.943407
9,processed food,product category,597,611,0.941347


In [10]:

trade_schemes = {
    "Общая схема": [
        "country",
        "organization",
        "agreement",
        "product",
        "regulation",
    ],
    "Специализированная схема": [
        "country",
        "international organization",
        "trade agreement",
        "trade restriction",
        "product category",
        "legal regulation",
    ],
    "Альтернативная схема": [
        "country",
        "government organization",
        "international trade treaty",
        "import or export restriction",
        "traded goods",
        "export control regulation",
    ],
}

results = {}

for scheme_name, scheme_labels in trade_schemes.items():
    predictions = model.predict_entities(
        trade_text,
        scheme_labels,
        threshold=0.3,
    )

    results[scheme_name] = predictions

    print(f"\n{scheme_name}")
    display(
        pd.DataFrame(
            predictions,
            columns=["text", "label", "score"],
        )
    )



Общая схема


,text,label,score
0,European Commission,organization,0.984853
1,Russian steel,product,0.870514
2,aluminum,product,0.771935
3,medical equipment,product,0.951379
4,agricultural fertilizers,product,0.937795
5,European manufacturers,organization,0.633032
6,Germany,country,0.999456
7,Poland,country,0.999318
8,India,country,0.999575
9,United Arab Emirates,country,0.999331



Специализированная схема


,text,label,score
0,European Commission,international organization,0.985581
1,Russian,country,0.742213
2,aluminum,product category,0.619940
3,medical equipment,product category,0.935581
4,agricultural fertilizers,product category,0.899749
5,European manufacturers,international organization,0.435840
6,Germany,country,0.999472
7,Poland,country,0.999307
8,India,country,0.999597
9,United Arab Emirates,country,0.999228



Альтернативная схема


,text,label,score
0,European Commission,government organization,0.986952
1,Russian,country,0.725789
2,steel,traded goods,0.371730
3,aluminum,traded goods,0.690337
4,medical equipment,traded goods,0.906771
5,agricultural fertilizers,traded goods,0.861157
6,European manufacturers,government organization,0.322874
7,Germany,country,0.999470
8,Poland,country,0.999278
9,India,country,0.999564


In [11]:

from collections import defaultdict

mentions = defaultdict(dict)

for scheme_name, predictions in results.items():
    for entity in predictions:
        span = (entity["start"], entity["end"])
        mentions[span][scheme_name] = entity["label"]

rows = []

for (start, end), scheme_results in mentions.items():
    row = {
        "Упоминание": trade_text[start:end],
    }

    for scheme_name in trade_schemes:
        row[scheme_name] = scheme_results.get(
            scheme_name, "—"
        )

    rows.append(row)

comparison = pd.DataFrame(rows)

# Оставляем упоминания, для которых набор
# обнаруживших их схем или назначенных типов различается.
scheme_columns = list(trade_schemes)

differences = comparison[
    comparison[scheme_columns].nunique(axis=1) > 1
]

display(differences)


,Упоминание,Общая схема,Специализированная схема,Альтернативная схема
0,European Commission,organization,international organization,government organization
1,Russian steel,product,—,—
2,aluminum,product,product category,traded goods
3,medical equipment,product,product category,traded goods
4,agricultural fertilizers,product,product category,traded goods
5,European manufacturers,organization,international organization,government organization
10,Comprehensive Economic Partnership Agreement,agreement,trade agreement,export control regulation
11,CEPA,agreement,trade agreement,international trade treaty
12,pharmaceutical products,product,product category,traded goods
13,processed food,product,product category,traded goods


### Визуализация разметки

Покажем найденные упоминания непосредственно в строке. Цвета служат только для выделения, но не кодируют правильность результата.

In [12]:
from html import escape
from IPython.display import HTML, display

def highlight_entities(source_text, found_entities):
    # Для наглядности используем непересекающиеся упоминания по возрастанию позиции.
    found_entities = sorted(found_entities, key=lambda e: (e["start"], -(e["end"] - e["start"])))
    pieces, pos = [], 0
    for ent in found_entities:
        start, end = ent["start"], ent["end"]
        if start < pos:  # пропускаем пересекающиеся интервалы
            continue
        pieces.append(escape(source_text[pos:start]))
        mention = escape(source_text[start:end])
        label = escape(ent["label"])
        pieces.append(f'<mark style="padding:3px;border-radius:4px">{mention} <small>({label})</small></mark>')
        pos = end
    pieces.append(escape(source_text[pos:]))
    return HTML('<div style="line-height:2;font-size:15px">' + ''.join(pieces) + '</div>')

display(highlight_entities(trade_text, trade_entities))

## 7. Самостоятельный эксперимент

Измените текст и типы сущностей. Сравните два набора меток и два порога, затем вручную классифицируйте ошибки: неверная граница, неправильный тип, пропуск, лишнее упоминание.

In [13]:

MY_TEXT = (
    "The pharmaceutical company BioNova signed a licensing "
    "agreement with MedTech Solutions to develop a new cancer "
    "treatment. The agreement includes an upfront payment "
    "of $120 million and additional milestone payments. "
    "However, the companies emphasized that the agreement "
    "does not constitute a merger or an acquisition. "
    "Clinical trials are expected to begin in Boston in 2028."
)

MY_LABELS = [
    "company",
    "licensing agreement",
    "financial payment",
    "medical treatment",
    "clinical trial",
    "city",
    "date",
]

MY_THRESHOLD = 0.3

my_result = model.predict_entities(
    MY_TEXT,
    MY_LABELS,
    threshold=MY_THRESHOLD,
)

display(highlight_entities(MY_TEXT, my_result))

display(
    pd.DataFrame(
        my_result,
        columns=["text", "label", "start", "end", "score"],
    )
)


,text,label,start,end,score
0,BioNova,company,27,34,0.992608
1,licensing agreement,licensing agreement,44,63,0.762228
2,MedTech Solutions,company,69,86,0.979120
3,cancer treatment,medical treatment,104,120,0.991426
4,upfront payment,financial payment,148,163,0.864360
5,$120 million,financial payment,167,179,0.991989
6,milestone payments,financial payment,195,213,0.826618
7,Clinical trials,clinical trial,316,331,0.985806
8,Boston,city,357,363,0.998566
9,2028,date,367,371,0.976461


## 9. Дополнительно: русский текст

Англоязычная модель из основного примера **не предназначена специально для русского языка**. Для корректного сравнения можно загрузить отдельно многоязычную модель.

In [14]:
multilingual_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
russian_text = "Россия и Индия обсудили новое торговое соглашение в Москве."
russian_labels = ["страна", "город", "торговое соглашение"]
russian_result = multilingual_model.predict_entities(russian_text, russian_labels, threshold=0.3)
display(pd.DataFrame(russian_result))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

,start,end,text,label,score
0,0,6,Россия,страна,0.979713
1,9,14,Индия,страна,0.977844
2,24,49,новое торговое соглашение,торговое соглашение,0.793608
3,52,58,Москве,город,0.985975


## Итоги

1. GLiNER получает на вход текст и перечень типов; для смены типов необязательно переобучать модель.
2. Выбор формулировок меток и порога влияет на распознавание.
3. Возвращаемые символьные границы позволяют сопоставлять предсказания с эталонной разметкой.
4. Для предметной схемы следует отдельно проверить согласованность типов и качество на размеченном корпусе.

**Материалы:** [библиотека GLiNER](https://pypi.org/project/gliner/), [карточка модели](https://huggingface.co/gliner-community/gliner_small-v2.5), [исходная статья](https://arxiv.org/abs/2311.08526).